# OAuth-Protected MCP Server Targets with Agent-Mediated Authentication

## Overview

This tutorial demonstrates how an **agent** can call AgentCore Gateway's 3LO MCP targets **on behalf of a user** using AgentCore Identity's federated token flow.

- **Inbound auth**: Agent gets a federated Cognito access token via AgentCore Identity (`USER_FEDERATION`)
- **Outbound auth**: Authorization Code Grant (3LO) for GitHub access

### Architecture

```
Agent → get-workload-access-token-for-jwt(user ID token) → workload token
      → get-resource-oauth2-token(USER_FEDERATION) → federated Cognito access token
      → Gateway (Bearer token, validated by allowedClients)
      → Gateway checks token vault → injects GitHub 3LO token → GitHub MCP server
```

### When to use this over PKCE

| | PKCE (01-pkce-github.ipynb) | Agent-Mediated (this notebook) |
|---|---|---|
| Client secret | Not needed | Required |
| Browser login | Every session | First time only |
| Token refresh | Manual | Automatic (AgentCore Identity) |
| Best for | CLI testing, IDE integration | Production agents, SPA backends |

### Prerequisites

- AWS credentials configured
- GitHub OAuth App created (Client ID + Client Secret)
- Python 3.10+

In [ ]:
!pip install -q boto3 requests

In [ ]:
import boto3
import json
import time
import os
import sys
import requests
import subprocess
import hashlib
import hmac
import base64
import urllib.parse
import webbrowser
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
REGION = os.environ.get('AWS_REGION', 'us-west-2')

print(f"Region: {REGION}")
print(f"Timestamp: {timestamp}")
print("✓ Libraries imported")

In [ ]:
# === CONFIGURATION ===

# GitHub OAuth App credentials (from https://github.com/settings/developers)
# These are needed ONLY at setup time to register the GitHub credential provider
# with AgentCore Identity. The agent does NOT use these at runtime — AgentCore
# Identity stores them securely and handles the OAuth exchange with GitHub.
GITHUB_CLIENT_ID = ""       # From GitHub OAuth App
GITHUB_CLIENT_SECRET = ""   # From GitHub OAuth App

GATEWAY_NAME = f"oauth-3lo-agent-{timestamp}"
MCP_VERSION = "2025-11-25"
CALLBACK_URL = "http://localhost:3000/callback"

# Cognito settings (confidential client — secret managed by AgentCore Identity)
COGNITO_POOL_NAME = f"oauth-3lo-agent-pool-{timestamp}"
COGNITO_DOMAIN_PREFIX = f"oauth-3lo-agent-{timestamp}"
TEST_USER_EMAIL = "testuser@example.com"
TEST_USER_PASSWORD = "TestPass1!"

assert GITHUB_CLIENT_ID, "Set GITHUB_CLIENT_ID"
assert GITHUB_CLIENT_SECRET, "Set GITHUB_CLIENT_SECRET"

print("✓ Configuration set")


In [ ]:
cognito = boto3.client('cognito-idp', region_name=REGION)
agentcore_cp = boto3.client('bedrock-agentcore-control', region_name=REGION)
agentcore_dp = boto3.client('bedrock-agentcore', region_name=REGION)
state = {}
print("✓ AWS clients initialized")

## Step 1: Create Cognito User Pool

For the agent-mediated flow, we need a **confidential client** (with secret) for the agent, plus a Cognito credential provider registered in AgentCore Identity.

In [ ]:
# Create User Pool
pool = cognito.create_user_pool(
    PoolName=COGNITO_POOL_NAME,
    AutoVerifiedAttributes=['email'], UsernameAttributes=['email'],
    Policies={'PasswordPolicy': {'MinimumLength': 8, 'RequireUppercase': True, 'RequireLowercase': True, 'RequireNumbers': True, 'RequireSymbols': True}},
    Schema=[{'Name': 'email', 'AttributeDataType': 'String', 'Required': True, 'Mutable': True}]
)
state['pool_id'] = pool['UserPool']['Id']

# Domain
cognito.create_user_pool_domain(UserPoolId=state['pool_id'], Domain=COGNITO_DOMAIN_PREFIX)
state['cognito_domain'] = COGNITO_DOMAIN_PREFIX

# Confidential client (with secret)
conf_client = cognito.create_user_pool_client(
    UserPoolId=state['pool_id'],
    ClientName=f"confidential-client-{timestamp}",
    GenerateSecret=True,
    ExplicitAuthFlows=['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH', 'ALLOW_USER_SRP_AUTH'],
    SupportedIdentityProviders=['COGNITO'],
    CallbackURLs=[CALLBACK_URL],
    AllowedOAuthFlows=['code', 'implicit'],
    AllowedOAuthScopes=['openid', 'email', 'profile'],
    AllowedOAuthFlowsUserPoolClient=True
)
state['client_id'] = conf_client['UserPoolClient']['ClientId']
state['client_secret'] = conf_client['UserPoolClient']['ClientSecret']

# Test user
try:
    cognito.admin_create_user(
        UserPoolId=state['pool_id'], Username=TEST_USER_EMAIL,
        UserAttributes=[{'Name': 'email', 'Value': TEST_USER_EMAIL}, {'Name': 'email_verified', 'Value': 'true'}],
        TemporaryPassword=TEST_USER_PASSWORD, MessageAction='SUPPRESS'
    )
    cognito.admin_set_user_password(UserPoolId=state['pool_id'], Username=TEST_USER_EMAIL, Password=TEST_USER_PASSWORD, Permanent=True)
except: pass

state['discovery_url'] = f"https://cognito-idp.{REGION}.amazonaws.com/{state['pool_id']}/.well-known/openid-configuration"
print(f"✓ Pool: {state['pool_id']}")
print(f"✓ Client: {state['client_id']} (confidential)")
print(f"✓ User: {TEST_USER_EMAIL}")

## Step 2: Create Gateway + Cognito Delegation Provider

Register Cognito as an OAuth2 credential provider in AgentCore Identity (`CognitoOauth2` vendor). This lets the agent call `get-resource-oauth2-token` to get a federated Cognito access token for the user.

In [ ]:
# Create IAM role for Gateway with scoped permissions
iam = boto3.client('iam', region_name=REGION)
role_name = f'AgentCoreGatewayRole-{timestamp}'

trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'bedrock-agentcore.amazonaws.com'},
        'Action': 'sts:AssumeRole'
    }]
}

# Scoped policy for gateway operations
gateway_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Sid': 'AgentCoreGatewayAccess',
            'Effect': 'Allow',
            'Action': [
                'bedrock-agentcore:InvokeMcpTool',
                'bedrock-agentcore:GetResourceOauth2Token',
                'bedrock-agentcore:CompleteResourceTokenAuth',
                'bedrock-agentcore:GetWorkloadAccessToken',
                'bedrock-agentcore:GetWorkloadAccessTokenForJWT'
            ],
            'Resource': '*'
        },
        {
            'Sid': 'SecretsManagerAccess',
            'Effect': 'Allow',
            'Action': [
                'secretsmanager:GetSecretValue'
            ],
            'Resource': 'arn:aws:secretsmanager:*:*:secret:bedrock-agentcore-identity*'
        }
    ]
}

try:
    role = iam.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='IAM role for AgentCore Gateway with scoped permissions'
    )
    GATEWAY_ROLE_ARN = role['Role']['Arn']
    iam.put_role_policy(
        RoleName=role_name,
        PolicyName='AgentCoreGatewayPolicy',
        PolicyDocument=json.dumps(gateway_policy)
    )
    print(f'✓ Created IAM role: {GATEWAY_ROLE_ARN}')
except iam.exceptions.EntityAlreadyExistsException:
    GATEWAY_ROLE_ARN = iam.get_role(RoleName=role_name)['Role']['Arn']
    print(f'✓ IAM role exists: {GATEWAY_ROLE_ARN}')

state['role_name'] = role_name
# Wait for IAM propagation
time.sleep(10)

In [ ]:
# Create gateway
gw = agentcore_cp.create_gateway(
    name=GATEWAY_NAME,
    description='OAuth-protected MCP targets with agent-mediated inbound auth',
    roleArn=GATEWAY_ROLE_ARN,
    protocolType='MCP',
    protocolConfiguration={'mcp': {'supportedVersions': [MCP_VERSION]}},
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration={
        'customJWTAuthorizer': {
            'discoveryUrl': state['discovery_url'],
            'allowedClients': [state['client_id']]
        }
    }
)
state['gateway_id'] = gw['gatewayId']

for _ in range(24):
    status = agentcore_cp.get_gateway(gatewayIdentifier=state['gateway_id'])['status']
    if status == 'READY': break
    time.sleep(5)

gw_info = agentcore_cp.get_gateway(gatewayIdentifier=state['gateway_id'])
state['gateway_url'] = gw_info['gatewayUrl']
print(f"✓ Gateway: {state['gateway_url']}")

In [ ]:
# Register Cognito as credential provider in AgentCore Identity
cognito_provider = agentcore_cp.create_oauth2_credential_provider(
    name=f"cognito-delegation-{timestamp}",
    credentialProviderVendor='CognitoOauth2',
    oauth2ProviderConfigInput={
        'includedOauth2ProviderConfig': {
            'clientId': state['client_id'],
            'clientSecret': state['client_secret'],
            'issuer': f"https://cognito-idp.{REGION}.amazonaws.com/{state['pool_id']}",
            'authorizationEndpoint': f"https://{state['cognito_domain']}.auth.{REGION}.amazoncognito.com/oauth2/authorize",
            'tokenEndpoint': f"https://{state['cognito_domain']}.auth.{REGION}.amazoncognito.com/oauth2/token"
        }
    }
)
state['cognito_provider_name'] = f"cognito-delegation-{timestamp}"
cognito_callback = cognito_provider['callbackUrl']
print(f"✓ Cognito delegation provider: {state['cognito_provider_name']}")

# Add AgentCore callback URL to Cognito client
cognito.update_user_pool_client(
    UserPoolId=state['pool_id'], ClientId=state['client_id'],
    CallbackURLs=[CALLBACK_URL, cognito_callback],
    AllowedOAuthFlows=['code', 'implicit'],
    AllowedOAuthScopes=['openid', 'email', 'profile'],
    AllowedOAuthFlowsUserPoolClient=True,
    SupportedIdentityProviders=['COGNITO'],
    ExplicitAuthFlows=['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH', 'ALLOW_USER_SRP_AUTH']
)
print(f"✓ Added AgentCore callback to Cognito client")

# Create standalone workload identity
workload_name = f"agent-workload-{timestamp}"
agentcore_cp.create_workload_identity(
    name=workload_name,
    allowedResourceOauth2ReturnUrls=[CALLBACK_URL]
)
state['workload_name'] = workload_name
print(f"✓ Workload identity: {workload_name}")

## Step 3: Create GitHub Provider + Target

Same as the PKCE notebook — register GitHub and create the 3LO MCP server target.

In [ ]:
# Start callback server for admin auth (target creation may need it)
callback_proc = subprocess.Popen(
    [sys.executable, 'oauth2_callback_server.py', REGION, '3000', '/tmp/gateway-jwt'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(1)

# GitHub provider
provider = agentcore_cp.create_oauth2_credential_provider(
    name=f"github-3lo-{timestamp}",
    credentialProviderVendor='GithubOauth2',
    oauth2ProviderConfigInput={'githubOauth2ProviderConfig': {'clientId': GITHUB_CLIENT_ID, 'clientSecret': GITHUB_CLIENT_SECRET}}
)
state['github_provider_name'] = f"github-3lo-{timestamp}"
state['github_provider_arn'] = provider['credentialProviderArn']
state['github_callback_url'] = provider['callbackUrl']
print(f"✓ GitHub provider: {state['github_provider_name']}")
print(f"⚠ Update GitHub OAuth App callback URL to: {state['github_callback_url']}")

# GitHub target
target = agentcore_cp.create_gateway_target(
    gatewayIdentifier=state['gateway_id'],
    name=f"github-mcp-3lo-{timestamp}",
    description='GitHub MCP server with 3LO',
    targetConfiguration={'mcp': {'mcpServer': {'endpoint': 'https://api.githubcopilot.com/mcp'}}},
    credentialProviderConfigurations=[{
        'credentialProviderType': 'OAUTH',
        'credentialProvider': {'oauthCredentialProvider': {
            'providerArn': state['github_provider_arn'],
            'grantType': 'AUTHORIZATION_CODE',
            'defaultReturnUrl': CALLBACK_URL,
            'scopes': ['repo', 'user', 'workflow']
        }}
    }]
)
state['github_target_id'] = target['targetId']
state['github_target_name'] = f"github-mcp-3lo-{timestamp}"
print(f"✓ Target: {state['github_target_id']} (status: {target['status']})")

# If CREATE_PENDING_AUTH, complete via console
if target['status'] == 'CREATE_PENDING_AUTH':
    print("")
    print("=" * 60)
    print("⚠️  ACTION REQUIRED: Admin Authorization Needed")
    print("=" * 60)
    print(f"")
    print(f"The target is in CREATE_PENDING_AUTH status.")
    print(f"You MUST authorize it in the AgentCore console before continuing.")
    print(f"")
    print(f"1. Go to the AgentCore console")
    print(f"2. Navigate to Gateway: {state['gateway_id']}")
    print(f"3. Click on the target and click 'Authorize'")
    print(f"4. Complete the OAuth consent in the browser")
    print(f"5. Wait for the target status to change to READY")
    print(f"")
    print(f"⚠️  DO NOT proceed to the next cell until the target is READY.")
    print("=" * 60)


## Step 4: Agent Gets Federated Token

The agent:
1. Gets the user's Cognito ID token (from their login session)
2. Calls `get-workload-access-token-for-jwt` to get a user-scoped workload token
3. Calls `get-resource-oauth2-token` with `USER_FEDERATION` to get a federated Cognito access token

The federated token has `client_id` + `sub` — the gateway validates `client_id` and uses `sub` for the token vault lookup.

In [ ]:
# Step A: Get user's Cognito ID token (simulates user login)
secret_hash = base64.b64encode(
    hmac.new(state['client_secret'].encode(), (TEST_USER_EMAIL + state['client_id']).encode(), hashlib.sha256).digest()
).decode()

auth_result = cognito.initiate_auth(
    ClientId=state['client_id'],
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME': TEST_USER_EMAIL, 'PASSWORD': TEST_USER_PASSWORD, 'SECRET_HASH': secret_hash}
)
user_id_token = auth_result['AuthenticationResult']['IdToken']
print(f"✓ Got user ID token")

# Step B: Get workload token
workload_token = agentcore_dp.get_workload_access_token_for_jwt(
    workloadName=state['workload_name'],
    userToken=user_id_token
)['workloadAccessToken']
print(f"✓ Got workload token ({len(workload_token)} chars, opaque)")

# Step C: Get federated Cognito access token
# Start callback server for consent (if first time)
callback_proc = subprocess.Popen(
    [sys.executable, 'oauth2_callback_server.py', REGION, '3000', '/tmp/gateway-jwt'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(1)
with open('/tmp/gateway-jwt', 'w') as f:
    f.write(user_id_token)

result = agentcore_dp.get_resource_oauth2_token(
    workloadIdentityToken=workload_token,
    resourceCredentialProviderName=state['cognito_provider_name'],
    scopes=['openid', 'email', 'profile'],
    oauth2Flow='USER_FEDERATION',
    resourceOauth2ReturnUrl=CALLBACK_URL
)

if 'authorizationUrl' in result:
    print("⚠ Cognito consent required (first time)")
    print(f"  Login as: {TEST_USER_EMAIL} / {TEST_USER_PASSWORD}")
    webbrowser.open(result['authorizationUrl'])
    input("  Press Enter after consent...")
    
    # Retry
    workload_token = agentcore_dp.get_workload_access_token_for_jwt(
        workloadName=state['workload_name'], userToken=user_id_token
    )['workloadAccessToken']
    result = agentcore_dp.get_resource_oauth2_token(
        workloadIdentityToken=workload_token,
        resourceCredentialProviderName=state['cognito_provider_name'],
        scopes=['openid', 'email', 'profile'],
        oauth2Flow='USER_FEDERATION',
        resourceOauth2ReturnUrl=CALLBACK_URL
    )

ACCESS_TOKEN = result['accessToken']

# Decode
payload = ACCESS_TOKEN.split('.')[1]
payload += '=' * (4 - len(payload) % 4)
claims = json.loads(base64.urlsafe_b64decode(payload))
print(f"\n✓ Got federated access token")
print(f"  sub:       {claims['sub']}")
print(f"  client_id: {claims['client_id']}")
print(f"  token_use: {claims['token_use']}")

## Step 5: Call Gateway → GitHub

Same as PKCE — the gateway doesn't know which auth method was used. It just validates the JWT.

> **Note:** If no tools are returned, the target may still be syncing. Wait a few minutes and retry — it can take some time for the gateway to discover and cache the tools from the MCP server.

In [ ]:
headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Authorization': f'Bearer {ACCESS_TOKEN}',
    'Mcp-Protocol-Version': MCP_VERSION
}

# Initialize
requests.post(state['gateway_url'], headers=headers, json={
    'jsonrpc': '2.0', 'id': 1, 'method': 'initialize',
    'params': {'protocolVersion': MCP_VERSION, 'capabilities': {}, 'clientInfo': {'name': 'agent', 'version': '1.0'}}
})

# Save federated token for 3LO binding
with open('/tmp/gateway-jwt', 'w') as f:
    f.write(ACCESS_TOKEN)

# Call get_me
tool_name = f"{state['github_target_name']}___get_me"
resp = requests.post(state['gateway_url'], headers=headers, json={
    'jsonrpc': '2.0', 'id': 2, 'method': 'tools/call',
    'params': {'name': tool_name, 'arguments': {}}
})
result = resp.json()

if result.get('error', {}).get('code') == -32042:
    url = result['error']['data']['elicitations'][0]['url']
    print("⚠ GitHub consent required")
    webbrowser.open(url)
    input("  Press Enter after authorizing on GitHub...")
    time.sleep(2)
    resp = requests.post(state['gateway_url'], headers=headers, json={
        'jsonrpc': '2.0', 'id': 3, 'method': 'tools/call',
        'params': {'name': tool_name, 'arguments': {}}
    })
    result = resp.json()

if 'result' in result:
    data = json.loads(result['result']['content'][0]['text'])
    print(f"\n✓ GitHub get_me succeeded!")
    print(json.dumps(data, indent=2))
else:
    print(f"✖ Error: {result}")

callback_proc.terminate()
os.remove('/tmp/gateway-jwt')

## Step 6: Cleanup

In [ ]:
for name, fn in [
    ('target', lambda: agentcore_cp.delete_gateway_target(gatewayIdentifier=state['gateway_id'], targetId=state['github_target_id'])),
    ('gateway', lambda: (time.sleep(5), agentcore_cp.delete_gateway(gatewayIdentifier=state['gateway_id']))),
    ('github provider', lambda: agentcore_cp.delete_oauth2_credential_provider(name=state['github_provider_name'])),
    ('cognito provider', lambda: agentcore_cp.delete_oauth2_credential_provider(name=state['cognito_provider_name'])),
    ('workload', lambda: agentcore_cp.delete_workload_identity(name=state['workload_name'])),
    ('cognito domain', lambda: cognito.delete_user_pool_domain(UserPoolId=state['pool_id'], Domain=state['cognito_domain'])),
    ('cognito pool', lambda: cognito.delete_user_pool(UserPoolId=state['pool_id']))
]:
    try:
        fn()
        print(f"✓ Deleted {name}")
    except Exception as e:
        print(f"⚠ {name}: {e}")

print("\n✓ Cleanup complete")

# Delete IAM role
try:
    iam.delete_role_policy(RoleName=state['role_name'], PolicyName='AgentCoreGatewayPolicy')
    iam.delete_role(RoleName=state['role_name'])
    print(f"✓ Deleted IAM role: {state['role_name']}")
except Exception as e:
    print(f"⚠ IAM role: {e}")
